In [ ]:
# Run once in Colab
!pip install -q wandb pandas

In [ ]:
import wandb
wandb.login()

In [ ]:
import re
import pandas as pd

ENTITY  = "arc_agi"
PROJECT = "slm-instability"

api  = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

rows = []
for run in runs:
    name = run.name or ""
    if not name.startswith("lr-sweep-"):
        continue

    summary = run.summary

    # LR: prefer resolved value written to summary by WandBCallback
    # (reflects CONFIG_OVERRIDES_JSON); fall back to parsing run name
    lr = summary.get("config/lr")
    if lr is None:
        m = re.search(r"-lr(.+)$", name)
        lr = float(m.group(1)) if m else None

    # model size from summary (written by WandBCallback) or run name
    num_layers = summary.get("config/num_layers")
    model_dim  = summary.get("config/model_dim")
    m = re.search(r"lr-sweep-(.+)-lr", name)
    model_name = m.group(1) if m else name  # e.g. 2p4M, 9p4M, 19M, 42M

    rows.append({
        "run_name":      name,
        "model":         model_name,
        "num_layers":    num_layers,
        "model_dim":     model_dim,
        "lr":            lr,
        "state":         run.state,
        "best_val_loss": summary.get("best_val_loss"),
        "last_train_loss": summary.get("last_train_loss"),
        "failed":        summary.get("failed", False),
    })

df = pd.DataFrame(rows).sort_values(["model_dim", "num_layers", "lr"]).reset_index(drop=True)
print(df)